# WeightedKgBlend — DistMult
Reads splits from `weightedkgblend-splits` kernel. Set `SPLITS_TO_RUN=[0]` to benchmark one split first.


In [ ]:
import subprocess, sys
for p in ['pykeen>=1.10.0','optuna>=3.0.0','scipy>=1.10.0','scikit-learn>=1.3.0']:
    subprocess.run([sys.executable,'-m','pip','install',p,'--quiet'], check=True)
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import torch
MODEL_NAME     = 'DistMult'
INDICATION_REL = 'indication'
N_SPLITS       = 5
EMBEDDING_DIM  = 128
NUM_EPOCHS     = 300
BATCH_SIZE     = 512
PATIENCE       = 15
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_SEED    = 42

# ── Run only these split indices. Change to list(range(5)) for all splits. ──
SPLITS_TO_RUN  = [0]   # start with just slice_0 to benchmark timing

print(f'Model        : {MODEL_NAME}')
print(f'Device       : {DEVICE}')
print(f'Splits to run: {SPLITS_TO_RUN}')


In [ ]:
from pathlib import Path
import pandas as pd

# Splits come from the weightedkgblend-splits kernel output
SPLITS = Path('/kaggle/input/weightedkgblend-splits/splits')
WORK   = Path('/kaggle/working')
WORK.mkdir(exist_ok=True)

if not SPLITS.exists():
    raise FileNotFoundError(f"Splits not found at {SPLITS}. Make sure the weightedkgblend-splits kernel is attached as a data source.")

print("Splits found:")
for sl in sorted(SPLITS.iterdir()):
    files = [f.name for f in sl.iterdir()]
    print(f"  {sl.name}: {files}")


In [ ]:
import torch, numpy as np, pandas as pd
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline

def get_preds(model, factory, relation, device):
    model.eval()
    rel_id = factory.relation_to_id.get(relation)
    if rel_id is None: return pd.DataFrame()
    triples = factory.mapped_triples
    mask = triples[:,1] == rel_id
    rows = []
    with torch.no_grad():
        for triple in triples[mask]:
            h,r,t = triple[0].item(), triple[1].item(), triple[2].item()
            hr = torch.tensor([[h,r]], device=device)
            scores = model.score_t(hr).squeeze(0)
            order  = torch.argsort(scores, descending=True).cpu().numpy()
            rank   = int(np.where(order==t)[0][0]) + 1
            rows.append({'drug': factory.entity_id_to_label[h],
                         'expected_disease': factory.entity_id_to_label[t],
                         'rank': rank, 'reciprocal_rank': 1.0/rank})
    return pd.DataFrame(rows)


In [ ]:
import time

PRED_DIR = WORK / 'predictions' / 'DistMult'
PRED_DIR.mkdir(parents=True, exist_ok=True)

total_start = time.time()

for i in SPLITS_TO_RUN:
    sl     = SPLITS / f'slice_{i}'
    outdir = PRED_DIR / f'slice_{i}'
    outdir.mkdir(parents=True, exist_ok=True)

    if (outdir / 'predictions_test.tsv').exists():
        print(f'SKIP DistMult/slice_{i} (already done)')
        continue

    print(f'\n==================================================')
    print(f'Training DistMult — slice_{i}')
    print(f'==================================================')
    t0 = time.time()

    tf_train = TriplesFactory.from_labeled_triples(
        pd.read_csv(sl/'kge_train.tsv', sep='\t', header=None, names=['h','r','t']).values.astype(str))
    tf_test  = TriplesFactory.from_labeled_triples(
        pd.read_csv(sl/'ind_test.tsv',  sep='\t', header=None, names=['h','r','t']).values.astype(str),
        entity_to_id=tf_train.entity_to_id, relation_to_id=tf_train.relation_to_id)
    tf_valid = TriplesFactory.from_labeled_triples(
        pd.read_csv(sl/'ind_valid.tsv', sep='\t', header=None, names=['h','r','t']).values.astype(str),
        entity_to_id=tf_train.entity_to_id, relation_to_id=tf_train.relation_to_id)

    print(f'  Train triples : {len(tf_train.mapped_triples):,}')
    print(f'  Entities      : {tf_train.num_entities:,}')

    res = pipeline(
        training=tf_train, testing=tf_test, validation=tf_valid,
        model='DistMult',
        model_kwargs=dict(embedding_dim=EMBEDDING_DIM),
        optimizer='Adam', optimizer_kwargs=dict(lr=0.001),
        training_kwargs=dict(num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE),
        stopper='early', stopper_kwargs=dict(patience=PATIENCE),
        device=DEVICE, random_seed=RANDOM_SEED)

    for split_name, tf in [('test', tf_test), ('valid', tf_valid)]:
        preds = get_preds(res.model, tf, INDICATION_REL, DEVICE)
        preds.to_csv(outdir / f'predictions_{split_name}.tsv', sep='\t', index=False)
        mrr = preds.reciprocal_rank.mean()
        print(f'  {split_name} MRR: {mrr:.4f}')

    elapsed = time.time() - t0
    print(f'\n  slice_{i} done in {elapsed/60:.1f} min')

total_elapsed = time.time() - total_start
print(f'\nTotal elapsed: {total_elapsed/60:.1f} min')
print(f'Estimated time for all 5 splits: {total_elapsed/60 * 5 / len(SPLITS_TO_RUN):.1f} min')
